# Process Data For An OCT Melting Experiment (Time-series of Volumes and Camera Images)

In [1]:
#%% imports
from pathlib import Path
# import json
# import configparser
import pprint
pp = pprint.PrettyPrinter(indent=4);
import time
import math
from collections import namedtuple

# # pip install slicerio
# import slicerio.server

# # packages for 3d
# #   # for OCT 3D stuff and using in jupyter
# #   - trame-jupyter-extension
# #   - trame
# #   - trame-vtk
# #   - trame-vuetify
# #   - ipywidgets

import matplotlib.pyplot as plt
import plotly.express as px

# interactive panels
import panel as pn
pn.extension('plotly');

import numpy as np
import pandas as pd
import addict

import cv2
import skimage
import scipy.signal

import pyvista as pv
from PIL import Image
import SimpleITK as sitk

# with vedo
#from vedo import dataurl, Volume, Text2D
import vedo
vedo.settings.default_backend = 'vtk'
#from vedo.applications import Slicer3DPlotter
import naatos_oct_tools.plotters.vedo_plotters as vedo_plotters
import naatos_oct_tools.plotters.sitk_plotters as sitk_plotters

import naatos_oct_tools.thorlabs_oct_file_reading
import naatos_oct_tools.oct_linear_scan_processing

import papermill

In [2]:
# Magics to autoreload submodules when they are modified
%load_ext autoreload
%autoreload 2

In [7]:
#%% Load OCT study information
octstudies = [];
folder_octexport_root = Path(r'\\file.corp.ghlabs.org\Shared\Projects\NAATOS\V1\NAATOS_OCT_WORK\OCTExport')
#folder_octexport_root = Path(r'D:\SGProjects\NAATOS\OCTlocal')

# Study list to process
studylist = [
    #'WaxMeltingStudy_20250701_test1',
    'WaxMeltingStudy_20250702_test1',
]

# Load list of data
for studyname in studylist:
    octstudy = naatos_oct_tools.thorlabs_oct_file_reading.OCT_Study_Folder(studyname,folder_octexport_root);
    octstudies.append(octstudy);


STUDY: WaxMeltingStudy_20250702_test1
{   'study_has_an_ini_file': False,
    'study_has_json_info_file': False,
    'study_has_yat_log': False,
    'study_num_jpg_files': 0,
    'study_num_oct_files': 231,
    'study_num_vtk_files': 0}


In [ ]:
#%% Load OCTSTUDY object, and start some processing on it
for idx,octstudy in enumerate(octstudies):
    octstudy : naatos_oct_tools.thorlabs_oct_file_reading.OCT_Study_Folder;
    print(octstudy)

    #fname_merged_and_rescaled_volume = octstudy.folder_study_processed/'{:s}_STACKED_RESCALED_{:}to{:}_uint8.vtk'.format(octstudy.name,oct_scalar_min,oct_scalar_max);
    fname_merged_and_rescaled_volume = octstudy.folder_study_processed/'{:s}_STACKED_RESCALED.vtk'.format(octstudy.name);
    
    if(octstudy.study_info['study_num_oct_files']>0):
        # --Loading And Pre-Processing--
        if(fname_merged_and_rescaled_volume.exists()):
            pass;
            #print(f'Loading {fname_merged_and_rescaled_volume.name}')
            # Load the strip and merge into one volume
            #vdvol = vedo.Volume(pv.read(fname_merged_and_rescaled_volume));
            #octstudy.vdvol = vdvol;
        else:
            if True:
                # Load OCT Data for this study
                octstudy.load_all_octs();


                # THESE WILL DO NOTHING IF ANTICIPATED OUTPUTS/ARTIFACTS ALREADY EXIST IN THE PROCESSED FOLDER

                # RGB Camera Images - Write them out as .jpg to processed folder
                naatos_oct_tools.oct_linear_scan_processing.process_rgbcamera_and_make_individual_images(octstudy);

                # # RGB Camera Images - Make a montage and write out as .jpg to processed folder
                # naatos_oct_tools.oct_linear_scan_processing.process_rgbcamera_and_make_montage_image(octstudy);
                #break;

            if False:
                # IF MERGED STACKED AND RESCALED TO SCALAR RANGE FILE EXISTS, LOAD THAT; OTHERWISE PROCESS IT HERE
                octstudy.folder_study_processed.mkdir(exist_ok=True);
                fname = fname_merged_and_rescaled_volume;
                if(fname.exists()):
                    print('Stacked volume byte-size .vtk file already exists, will not recreate.');
                    print(fname);
                else:
                    print(f'Generating merged and rescaled .vtk volume');
                    # Generate merged and rescaled volume
                    vdvol = octstudy.generate_merged_vdvol_and_rescaled(oct_scalar_min,oct_scalar_max);
                    octstudy.vdvol = vdvol;
                
                    # save this byte-adjusted volume
                    vdvol.dataset.save(fname);
            
                # Unload OCT Data for this study (we will still keep the vdvol)
                octstudy.unload_all_octdata();

        # # --Detailed Image Processing--
        # if hasattr(octstudy,'vdvol'):
        #     del octstudy.vdvol;

<OCT_Study_Folder Object>
WaxMeltingStudy_20250702_test1 in folder //file.corp.ghlabs.org/Shared/Projects/NAATOS/V1/NAATOS_OCT_WORK/OCTExport
{   'study_has_an_ini_file': False,
    'study_has_json_info_file': False,
    'study_has_yat_log': False,
    'study_num_jpg_files': 0,
    'study_num_oct_files': 231,
    'study_num_vtk_files': 0}
Image Dims:(345, 175, 300) PixelSpacing[mm]:(0.003475, 0.02, 0.02)
Image Dims:(345, 175, 300) PixelSpacing[mm]:(0.003475, 0.02, 0.02)
Image Dims:(345, 175, 300) PixelSpacing[mm]:(0.003475, 0.02, 0.02)
Image Dims:(345, 175, 300) PixelSpacing[mm]:(0.003475, 0.02, 0.02)
Image Dims:(345, 175, 300) PixelSpacing[mm]:(0.003475, 0.02, 0.02)
Image Dims:(345, 175, 300) PixelSpacing[mm]:(0.003475, 0.02, 0.02)
Image Dims:(345, 175, 300) PixelSpacing[mm]:(0.003475, 0.02, 0.02)
Image Dims:(345, 175, 300) PixelSpacing[mm]:(0.003475, 0.02, 0.02)
Image Dims:(345, 175, 300) PixelSpacing[mm]:(0.003475, 0.02, 0.02)
Image Dims:(345, 175, 300) PixelSpacing[mm]:(0.003475, 0

# Join and merge using simpleitk into something that Slicer will interpret as a timeseries

In [16]:
assert(len(octstudies)==1);
octstudy = octstudies[0];
study_name = octstudy.name;

warning, the following step will take a lot of peak memory!

In [10]:
# save simpleitk concatenating the volumes, which Slicer will interpret as a volume time-sequence

if True:
    composer = sitk.ComposeImageFilter();
    simgjoined = composer.Execute([octdata._make_sitkvol() for octdata in octstudy.octdatalist]);

In [11]:
print(simgjoined)

VectorImage (000001C213973770)
  RTTI typeinfo:   class itk::VectorImage<float,3>
  Reference Count: 1
  Modified Time: 6377
  Debug: Off
  Object Name: 
  Observers: 
    none
  Source: (none)
  Source output name: (none)
  Release Data: Off
  Data Released: False
  Global Release Data: Off
  PipelineMTime: 6364
  UpdateMTime: 6376
  RealTimeStamp: 0 seconds 
  LargestPossibleRegion: 
    Dimension: 3
    Index: [0, 0, 0]
    Size: [345, 175, 300]
  BufferedRegion: 
    Dimension: 3
    Index: [0, 0, 0]
    Size: [345, 175, 300]
  RequestedRegion: 
    Dimension: 3
    Index: [0, 0, 0]
    Size: [345, 175, 300]
  Spacing: [0.003475, 0.02, 0.02]
  Origin: [0, 0, 0]
  Direction: 
1 0 0
0 1 0
0 0 1

  IndexToPointMatrix: 
0.003475 0 0
0 0.02 0
0 0 0.02

  PointToIndexMatrix: 
287.77 0 0
0 50 0
0 0 50

  Inverse Direction: 
1 0 0
0 1 0
0 0 1

  VectorLength: 231
  PixelContainer: 
    ImportImageContainer (000001C37E040F90)
      RTTI typeinfo:   class itk::ImportImageContainer<unsigned _

Write the output to a .seq.nrrd which slicer will understand is a multivolume timeseries when it is loaded

In [13]:
writer = sitk.ImageFileWriter()
writer.SetFileName(octstudy.folder_study_processed/'{:s}.seq.nrrd'.format(study_name));
writer.Execute(simgjoined);

# (util) delete some items from processed folder

In [ ]:
# Warning this can be destructive deleting processed data!!
if False:
    for idx,octstudy in enumerate(octstudies):
        print('~~~~~~~');
        print(octstudy.name);
        
        if False:
            fname = (octstudy.folder_study_processed/'along_strip_data_extracted.hdf5')
            if(fname.exists()):
                print('deleted',fname.name);
                fname.unlink();
            
            fname = (octstudy.folder_study_processed/'data_extracted.npz')
            if(fname.exists()):
                print('deleted',fname.name);
                fname.unlink();

            flist = octstudy.folder_study_processed.glob('figout*');
            for fname in flist:
                print('deleted',fname.name);
                fname.unlink();

            flist = octstudy.folder_study_processed.glob('processing_step*');
            for fname in flist:
                print('deleted',fname.name);
                fname.unlink();

        if True:
            # move files to a backup folder in the processed directory

            folder_backup = (octstudy.folder_study_processed/'_backup20250630');
            folder_backup.mkdir(exist_ok=True);

            fname = (octstudy.folder_study_processed/'along_strip_data_extracted.hdf5')
            if(fname.exists()):
                print('move to backup',fname.name);
                fname.rename(folder_backup/fname.name);
        
            fname = (octstudy.folder_study_processed/'data_extracted.npz')
            if(fname.exists()):
                print('move to backup',fname.name);
                fname.rename(folder_backup/fname.name);

            flist = octstudy.folder_study_processed.glob('figout*');
            for fname in flist:
                print('move to backup',fname.name);
                fname.rename(folder_backup/fname.name);

            flist = octstudy.folder_study_processed.glob('processing_step*');
            for fname in flist:
                print('move to backup',fname.name);
                fname.rename(folder_backup/fname.name);
        #break;


# Make .jpg files from the camera RGB images

In [14]:
naatos_oct_tools.oct_linear_scan_processing.process_rgbcamera_and_make_individual_images(octstudy);

WaxMeltingStudy_20250702_test1_videocamera_0000 20250702T112810
WaxMeltingStudy_20250702_test1_videocamera_0001 20250702T112815
WaxMeltingStudy_20250702_test1_videocamera_0002 20250702T112822
WaxMeltingStudy_20250702_test1_videocamera_0003 20250702T112828
WaxMeltingStudy_20250702_test1_videocamera_0004 20250702T112834
WaxMeltingStudy_20250702_test1_videocamera_0005 20250702T112840
WaxMeltingStudy_20250702_test1_videocamera_0006 20250702T112846
WaxMeltingStudy_20250702_test1_videocamera_0007 20250702T112852
WaxMeltingStudy_20250702_test1_videocamera_0008 20250702T112857
WaxMeltingStudy_20250702_test1_videocamera_0009 20250702T112903
WaxMeltingStudy_20250702_test1_videocamera_0010 20250702T112909
WaxMeltingStudy_20250702_test1_videocamera_0011 20250702T112915
WaxMeltingStudy_20250702_test1_videocamera_0012 20250702T112921
WaxMeltingStudy_20250702_test1_videocamera_0013 20250702T112927
WaxMeltingStudy_20250702_test1_videocamera_0014 20250702T112933
WaxMeltingStudy_20250702_test1_videocame

# Process OCT Camera's RGB Images

In [18]:
octdata = octstudy.octdatalist[0];

In [19]:
octdata.cfg_oct_probe['camerascalingx']

'48.33446705'

In [20]:
octdata.cfg_oct_probe['camerascalingy']

'48.66782401'

In [ ]:
# test reads
# import nrrd
# header = nrrd.read_header((folder_study_processed/'RGBSequence.seq.mrb').as_posix())

In [21]:
import slicerio.server
class slicerio_server_simon_wrapper():
    # slicer's rest HTTP API
    # Note, exec command must be turned on
    # https://slicer.readthedocs.io/en/latest/user_guide/modules/webserver.html
    
    # slicerio API
    # https://github.com/lassoan/slicerio/blob/main/slicerio/server.py

    # slicer APIs
    # https://slicer.readthedocs.io/en/latest/developer_guide/mrml_overview.html#mrml-scene

    # script repository
    # https://slicer.readthedocs.io/en/latest/developer_guide/script_repository.html
    # https://github.com/Slicer/Slicer/blob/main/Docs/developer_guide/script_repository/gui.md
    @staticmethod
    def exec(commandstring):
        api_url = f"http://127.0.0.1:{slicerio.server.SERVER_PORT}/slicer/exec"
        print(api_url,commandstring);
        response = slicerio.server.requests.get(api_url,params=dict(source=commandstring));
        #api_url+='?{:s}'.format(slicerio.server.requests.utils.quote(commandstring));
        #response = slicerio.server.requests.get(api_url);
        #response = slicerio.server.requests.get(api_url,params=dict(source=commandstring));
        slicerio.server._report_error(response)
        return response;
    # @staticmethod
    # def exec(commandstring):
    #     api_url = f"http://127.0.0.1:{slicerio.server.SERVER_PORT}/slicer/exec"
    #     print(api_url,commandstring);
    #     response = slicerio.server.requests.get(api_url,params=dict(source=commandstring));
    #     #api_url+='?{:s}'.format(slicerio.server.requests.utils.quote(commandstring));
    #     #response = slicerio.server.requests.get(api_url);
    #     #response = slicerio.server.requests.get(api_url,params=dict(source=commandstring));
    #     slicerio.server._report_error(response)
    #     return response;
    @staticmethod
    def version():
        api_url = f"http://127.0.0.1:{slicerio.server.SERVER_PORT}/slicer/system/version";
        response = slicerio.server.requests.get(api_url);
        slicerio.server._report_error(response)
        return response.json();

In [22]:
slicer_running = slicerio.server.is_server_running();
print('Slicer running?',slicer_running)
if not slicer_running:
    print('Starting slicer...')
    slicerio.server.start_server();
    slicer_running = slicerio.server.is_server_running();
    print('Slicer running?',slicer_running)

Slicer running? True


In [23]:
r = slicerio_server_simon_wrapper.version()
print(r)

{'applicationName': 'Slicer', 'applicationDisplayName': 'Slicer', 'applicationVersion': '5.8.1', 'releaseType': 'Stable', 'repositoryUrl': 'https://github.com/Slicer/Slicer', 'repositoryBranch': 'Slicer', 'revision': '33241', 'majorVersion': 5, 'minorVersion': 8, 'arch': 'amd64', 'os': 'win', 'isCustomMainApplication': False, 'mainApplicationName': 'Slicer', 'mainApplicationRepositoryUrl': 'https://github.com/Slicer/Slicer', 'mainApplicationRepositoryRevision': '11eaf62', 'mainApplicationRevision': '33241', 'mainApplicationMajorVersion': 5, 'mainApplicationMinorVersion': 8, 'mainApplicationPatchVersion': 1}


In [24]:
octstudy.study_info

{'study_has_yat_log': False,
 'study_num_oct_files': 231,
 'study_num_vtk_files': 0,
 'study_num_jpg_files': 0,
 'study_has_an_ini_file': False,
 'study_has_json_info_file': False}

In [26]:
import slicerio.server

#if(octstudy.study_info['study_num_oct_files']>=1 and octstudy.study_info['study_has_yat_log']):
if(octstudy.study_info['study_num_oct_files']>=1):
    # CREATE A TIME_MOSAIC OF RGB
    octstudy.folder_study_processed.mkdir(exist_ok=True);
    tmparrays = [];
    for count,octdata in enumerate(sorted(octstudy.octdatalist, key= lambda x: int(x.cfg_oct_xml.Ocity.Acquisition.Timestamp.etElem.text))):
        octdata_study = octdata.cfg_oct_xml.Ocity.MetaInfo.Study.etElem.text;
        octdata_timestr = time.strftime('%Y%m%dT%H%M%S',time.gmtime(int(octdata.cfg_oct_xml.Ocity.Acquisition.Timestamp.etElem.text)));
        
        #fname = '{:s}_{:02d}_{:s}'.format(octdata_study,count,octdata_timestr);
        fname = '{:s}_{:04d}'.format(octdata_study,count);
        print(fname,octdata_timestr);

        #octdata.volume.save(folder_study_processed/(fname+'.vtk'));
        #octdata.image.save(folder_study_processed/(fname+'.png'))


        simgcam = sitk.GetImageFromArray(np.array(octdata.image)[:,:,0:3],isVector=True);

        # camerascalingx and camerascalingy are defined as pixels/mm in the OCT probe .ini file
        simgcam.SetSpacing(( 1/float(octdata.cfg_oct_probe['camerascalingx']), 1/float(octdata.cfg_oct_probe['camerascalingy'])));

        # set orientation based on empirically determined orientation
        #simgcam.SetDirection(tuple([0,1,1,0]))
        #break;

        # write nrrd image
        writer = sitk.ImageFileWriter()
        #fname = octstudy.folder_study_processed/'{:s}_videocamera_{:04d}.nrrd'.format(study_name,count)
        fname = octstudy.folder_study_processed/'cam_{:04d}.nrrd'.format(count)
        writer.SetFileName(fname);
        writer.Execute(simgcam);

        # load image in slicer
        slicerio.server.file_load(fname)
        
        # delete image - # this is kinda important because slicer will assume files with _0001 suffixes all need to be loaded as zslices!
        Path(fname).unlink();
        #break;


WaxMeltingStudy_20250702_test1_0000 20250702T112810
WaxMeltingStudy_20250702_test1_0001 20250702T112815
WaxMeltingStudy_20250702_test1_0002 20250702T112822
WaxMeltingStudy_20250702_test1_0003 20250702T112828
WaxMeltingStudy_20250702_test1_0004 20250702T112834
WaxMeltingStudy_20250702_test1_0005 20250702T112840
WaxMeltingStudy_20250702_test1_0006 20250702T112846
WaxMeltingStudy_20250702_test1_0007 20250702T112852
WaxMeltingStudy_20250702_test1_0008 20250702T112857
WaxMeltingStudy_20250702_test1_0009 20250702T112903
WaxMeltingStudy_20250702_test1_0010 20250702T112909
WaxMeltingStudy_20250702_test1_0011 20250702T112915
WaxMeltingStudy_20250702_test1_0012 20250702T112921
WaxMeltingStudy_20250702_test1_0013 20250702T112927
WaxMeltingStudy_20250702_test1_0014 20250702T112933
WaxMeltingStudy_20250702_test1_0015 20250702T112940
WaxMeltingStudy_20250702_test1_0016 20250702T112946
WaxMeltingStudy_20250702_test1_0017 20250702T112952
WaxMeltingStudy_20250702_test1_0018 20250702T112958
WaxMeltingSt

In [ ]:
#simgcam.SetSpacing((float(octdata.cfg_oct_probe['camerascalingx']), float(octdata.cfg_oct_probe['camerascalingy'])))
#print(simgcam)

In [27]:
# slicer get list of volume nodes that were loaded
mrml_vector_volume_nodes = slicerio.server.node_ids(class_name="vtkMRMLVectorVolumeNode")
print(mrml_vector_volume_nodes)

['vtkMRMLVectorVolumeNode2', 'vtkMRMLVectorVolumeNode1', 'vtkMRMLVectorVolumeNode3', 'vtkMRMLVectorVolumeNode4', 'vtkMRMLVectorVolumeNode5', 'vtkMRMLVectorVolumeNode6', 'vtkMRMLVectorVolumeNode7', 'vtkMRMLVectorVolumeNode8', 'vtkMRMLVectorVolumeNode9', 'vtkMRMLVectorVolumeNode10', 'vtkMRMLVectorVolumeNode11', 'vtkMRMLVectorVolumeNode12', 'vtkMRMLVectorVolumeNode13', 'vtkMRMLVectorVolumeNode14', 'vtkMRMLVectorVolumeNode15', 'vtkMRMLVectorVolumeNode16', 'vtkMRMLVectorVolumeNode17', 'vtkMRMLVectorVolumeNode18', 'vtkMRMLVectorVolumeNode19', 'vtkMRMLVectorVolumeNode20', 'vtkMRMLVectorVolumeNode21', 'vtkMRMLVectorVolumeNode22', 'vtkMRMLVectorVolumeNode23', 'vtkMRMLVectorVolumeNode24', 'vtkMRMLVectorVolumeNode25', 'vtkMRMLVectorVolumeNode26', 'vtkMRMLVectorVolumeNode27', 'vtkMRMLVectorVolumeNode28', 'vtkMRMLVectorVolumeNode29', 'vtkMRMLVectorVolumeNode30', 'vtkMRMLVectorVolumeNode31', 'vtkMRMLVectorVolumeNode32', 'vtkMRMLVectorVolumeNode33', 'vtkMRMLVectorVolumeNode34', 'vtkMRMLVectorVolumeNo

In [28]:
# slicer make sequence node
# use Slicer exec rest endpoint to add sequence node
cmdstr = 'mergedSequenceNode = slicer.mrmlScene.AddNewNodeByClass("vtkMRMLSequenceNode", "RGB_Camera_Sequence")'.format();
#print(cmdstr)
slicerio_server_simon_wrapper.exec(cmdstr);

http://127.0.0.1:2016/slicer/exec mergedSequenceNode = slicer.mrmlScene.AddNewNodeByClass("vtkMRMLSequenceNode", "RGB_Camera_Sequence")


In [29]:
# slicer show the list of sequence nodes (should be only one)
mrml_sequence_nodes = slicerio.server.node_ids(class_name="vtkMRMLSequenceNode")
print(mrml_sequence_nodes);
assert(len(mrml_sequence_nodes)==1)

['vtkMRMLSequenceNode1']


In [30]:
# add frames to sequence
for count,mrml_vector_volume_node in enumerate(mrml_vector_volume_nodes):
        # use Slicer exec rest endpoint to add to sequence
        cmdstr = 'slicer.util.getNode("{:s}").SetDataNodeAtValue(slicer.util.getNode("{:s}"), "{:d}")'.format("vtkMRMLSequenceNode1",mrml_vector_volume_node,count);
        #print(cmdstr)
        slicerio_server_simon_wrapper.exec(cmdstr);

http://127.0.0.1:2016/slicer/exec slicer.util.getNode("vtkMRMLSequenceNode1").SetDataNodeAtValue(slicer.util.getNode("vtkMRMLVectorVolumeNode2"), "0")
http://127.0.0.1:2016/slicer/exec slicer.util.getNode("vtkMRMLSequenceNode1").SetDataNodeAtValue(slicer.util.getNode("vtkMRMLVectorVolumeNode1"), "1")
http://127.0.0.1:2016/slicer/exec slicer.util.getNode("vtkMRMLSequenceNode1").SetDataNodeAtValue(slicer.util.getNode("vtkMRMLVectorVolumeNode3"), "2")
http://127.0.0.1:2016/slicer/exec slicer.util.getNode("vtkMRMLSequenceNode1").SetDataNodeAtValue(slicer.util.getNode("vtkMRMLVectorVolumeNode4"), "3")
http://127.0.0.1:2016/slicer/exec slicer.util.getNode("vtkMRMLSequenceNode1").SetDataNodeAtValue(slicer.util.getNode("vtkMRMLVectorVolumeNode5"), "4")
http://127.0.0.1:2016/slicer/exec slicer.util.getNode("vtkMRMLSequenceNode1").SetDataNodeAtValue(slicer.util.getNode("vtkMRMLVectorVolumeNode6"), "5")
http://127.0.0.1:2016/slicer/exec slicer.util.getNode("vtkMRMLSequenceNode1").SetDataNodeAtVal

In [31]:
# set further sequence parameters
slicerio_server_simon_wrapper.exec('slicer.util.getNode("vtkMRMLSequenceNode1").SetIndexName("frame")')
slicerio_server_simon_wrapper.exec('slicer.util.getNode("vtkMRMLSequenceNode1").SetIndexUnit("")')

http://127.0.0.1:2016/slicer/exec slicer.util.getNode("vtkMRMLSequenceNode1").SetIndexName("frame")
http://127.0.0.1:2016/slicer/exec slicer.util.getNode("vtkMRMLSequenceNode1").SetIndexUnit("")


<Response [200]>

In [32]:
# remove volumes from slicer wworkspace
for mrml_vector_volume_node in mrml_vector_volume_nodes:
    slicerio.server.node_remove(id=mrml_vector_volume_node)

In [33]:
# slicer save scene
# use Slicer exec rest endpoint to save scene
cmdstr = 'slicer.util.saveScene("{:s}")'.format((octstudy.folder_study_processed/'RGB_Camera_Sequence.seq.mrb'.format()).as_posix());
#print(cmdstr)
slicerio_server_simon_wrapper.exec(cmdstr);

http://127.0.0.1:2016/slicer/exec slicer.util.saveScene("//file.corp.ghlabs.org/Shared/Projects/NAATOS/V1/NAATOS_OCT_WORK/OCTExport/WaxMeltingStudy_20250702_test1/processed/RGB_Camera_Sequence.seq.mrb")


In [ ]:
# save a transform
#tfm = sitk.Euler3DTransform();
tfm = sitk.AffineTransform(3);
#tfm.SetMatrix(np.array([1,0,0,0,  0,1,0,0, 0,0,1,0]))
#simgcam.SetDirection(tuple([0,1,1,0]))
#print(tfm)
#print(tfm.GetMatrix())
#tfm.SetMatrix(tuple([0,1,0,  0,0,-1,  1,0,0]))
tfm.SetMatrix(tuple([0,1,0,   0,0,-1,    -1,0,0]))

print(tfm)
sitk.WriteTransform(tfm, octstudy.folder_study_processed/'transform_rgb_camera.tfm')


itk::simple::AffineTransform
 AffineTransform (000001C21174D7F0)
   RTTI typeinfo:   class itk::AffineTransform<double,3>
   Reference Count: 1
   Modified Time: 56409
   Debug: Off
   Object Name: 
   Observers: 
     none
   Matrix: 
     0 1 0 
     0 0 -1 
     -1 0 0 
   Offset: [0, 0, 0]
   Center: [0, 0, 0]
   Translation: [0, 0, 0]
   Inverse: 
     0 0 -1 
     1 0 0 
     0 -1 0 
   Singular: 0



In [47]:
octdata.cfg_oct_probe

{'detailed description': '[Standard_OCTG, Handheld_OCTH, UserCustomizable_OCTP]',
 'description': 'Standard_OCTG_V1',
 'objective': 'LSM03_V1',
 'serialno': 'M01233227',
 'factorx': '0.726381',
 'offsetx': '-0.0149252',
 'factory': '0.705608',
 'offsety': '5.25506e-05',
 'quadraticfactorsx': '0,0,0,0,0,0,0,0,',
 'quadraticfactorsy': '0,0,0,0,0,0,0,0,',
 'referencestageoffset_mm': '0',
 'camerascalingx': '48.33446705',
 'camerascalingy': '48.66782401',
 'cameraangle': '89.79232727',
 'imagefield': '0.5338483453,0.0005434730556,0.008161109872,-0.003780778032,-0.0001346399076,0.001773982891,-1.335285833e-05,1.877583054e-05,-2.416062489e-05,-5.781046639e-05,1.36235883e-06,-2.262781436e-06,-3.47191218e-08,-1.570244649e-06,-1.528833423e-06,'}